In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML 


dt=1e-4 ; m=1.0 ; k=10.0 ; R=1.0 ; r0=0.5 ;      # cuentas de masa m ; muelle (k, longitud natural 2 r0) ; aro de radio R  (r0 < R)
tf=8; N=int(tf/dt);

t=np.linspace(0,tf,N)


om0  = np.sqrt(2*k/m)                       # ω0 = sqrt(2k/m)
om_c = om0*np.sqrt(1 - r0/R)                # velocidad angular crítica:  ρ(ω_c) = R   ->   ω_c² = ω0² (1 − r0/R)

om_a = 2.0                                  # ω < ω_c :  dos equilibrios estables ±z_eq  y  uno inestable (z = 0)
om_b = 4.0                                  # ω > ω_c :  un único equilibrio estable (z = 0)
rho_a = r0*om0**2/(om0**2 - om_a**2)        # ρ(ω)
zeq_a = np.sqrt(R**2 - rho_a**2)            # z_eq para ω = om_a

# Condiciones iniciales  (ω, z0, ż0)
def caso_peq(delta0):
    """Pequeñas oscilaciones alrededor de z_eq  (ω < ω_c):  z(0) = z_eq + δ0 ,  ż(0) = 0."""
    return dict(label=f'$\\delta_0={delta0}$', om=om_a, z=zeq_a + delta0, zd=0.0)

# SITUACIÓN 1: movimiento cualquiera (ω < ω_c): con energía suficiente para pasar de un pozo al otro
caso_general = dict(label='movimiento general ($\\omega<\\omega_c$)', om=om_a, z=0.5, zd=1.3)

# SITUACIÓN 2: pequeñas oscilaciones alrededor de z_eq. Los otros δ0 se usan para comparar con la solución analítica
# SITUACIÓN 3: régimen ω > ω_c  (oscilación alrededor de z = 0)
caso_sup = dict(label='$\\omega>\\omega_c$', om=om_b, z=0.5, zd=0.0)

casos = [caso_peq(0.03), caso_peq(0.0075), caso_peq(0.015), caso_peq(0.06), caso_general, caso_sup]
i_peq, i_gen, i_sup = 0, 4, 5                                # índices de las situaciones en 'casos'

## FUNCIONES GENERALES: LAGRANGIANO, HAMILTONIANO, EULER-LAGRANGE, HAMILTON Y PARÉNTESIS DE POISSON

In [ ]:
#FUNCIONES GENERALES (valen para cualquier sistema)
from types import SimpleNamespace

def lagrangian_hamiltonian(T, U, qs, qds, ps):
    """A partir de T y U devuelve:
         L = T − U                        (Lagrangiano)
         p_i = ∂L/∂q̇_i                    (momentos generalizados)
         H = Σ p_i q̇_i − L                (Hamiltoniano), SIEMPRE expresado en variables canónicas (q, p)
       T, U: expresiones de SymPy en (q, q̇) ;  qs, qds, ps: listas de símbolos (coordenadas, velocidades, momentos).
       Devuelve un objeto con L, p, H y, además, H_vel = H(q, q̇) y qdot = {q̇_i: q̇_i(q, p)}."""
    L = sp.simplify(T - U)
    p_expr = [sp.diff(L, qd) for qd in qds]                                           # p_i = ∂L/∂q̇_i  (derivadas parciales)
    H_vel  = sp.simplify(sum(p*qd for p, qd in zip(p_expr, qds)) - L)                 # H(q, q̇) = Σ p q̇ − L
    sol    = sp.solve([sp.Eq(P, pe) for P, pe in zip(ps, p_expr)], qds, dict=True)[0]  # se invierte p(q, q̇)  ->  q̇(q, p)
    H      = sp.simplify(H_vel.subs(sol))                                             # H(q, p)
    return SimpleNamespace(L=L, p=p_expr, H=H, H_vel=H_vel, qdot=sol)


def euler_lagrange(L, qs, qds, qdds, t=None):
    """Ecuaciones de Euler-Lagrange  d/dt(∂L/∂q̇_i) − ∂L/∂q_i = 0  para cualquier L(q, q̇ [, t]).
       qdds: símbolos para las aceleraciones q̈_i ; t: símbolo del tiempo si L depende explícitamente de él.
       Devuelve la lista de expresiones (cada una igualada a 0)."""
    n, EL = len(qs), []
    for i in range(n):
        p_i = sp.diff(L, qds[i])
        dp_dt = sum(sp.diff(p_i, qs[j])*qds[j] + sp.diff(p_i, qds[j])*qdds[j] for j in range(n))   # derivada total respecto a t
        if t is not None:
            dp_dt += sp.diff(p_i, t)
        EL.append(sp.simplify(dp_dt - sp.diff(L, qs[i])))
    return EL


def hamilton_equations(H, qs, ps):
    """Ecuaciones de Hamilton para cualquier H(q, p):  q̇_i = ∂H/∂p_i ,  ṗ_i = −∂H/∂q_i.  Devuelve (qdot_eq, pdot_eq)."""
    return [sp.diff(H, P) for P in ps], [-sp.diff(H, q) for q in qs]


def poisson_bracket(f, g, qs, ps):
    """Paréntesis de Poisson  {f, g} = Σ_i ( ∂f/∂q_i ∂g/∂p_i − ∂f/∂p_i ∂g/∂q_i )."""
    return sp.simplify(sum(sp.diff(f, q)*sp.diff(g, p) - sp.diff(f, p)*sp.diff(g, q) for q, p in zip(qs, ps)))

## (a), (b), (e) LAGRANGIANO, HAMILTONIANO Y ECUACIÓN DE MOVIMIENTO

In [ ]:
#(a), (b) Cinemática, Lagrangiano y Hamiltoniano   (con las funciones generales)
t_s = sp.symbols('t', real=True)
z_s, zd_s, zdd_s, p_s = sp.symbols('z zdot zddot p', real=True)
m_, k_, R_, r0_, om_, g_ = sp.symbols('m k R r0 omega g', positive=True)
zf = sp.Function('z')(t_s)

# Las dos cuentas tienen la misma z y están en lados opuestos del aro, que gira con φ = ω t :   r = sqrt(R² − z²)
rho_f = sp.sqrt(R_**2 - zf**2)
pos1 = sp.Matrix([ rho_f*sp.cos(om_*t_s),  rho_f*sp.sin(om_*t_s), zf])
pos2 = sp.Matrix([-rho_f*sp.cos(om_*t_s), -rho_f*sp.sin(om_*t_s), zf])
print('1 grado de libertad: 2 cuentas × 1 (posición en el aro) − 1 (igual z)  ->  coordenada generalizada z')

T_f   = sp.Rational(1,2)*m_*(pos1.diff(t_s).dot(pos1.diff(t_s)) + pos2.diff(t_s).dot(pos2.diff(t_s)))
T_sym = sp.simplify(T_f.subs(zf.diff(t_s), zd_s).subs(zf, z_s))
T2 = m_*R_**2*zd_s**2/(R_**2 - z_s**2)                              # parte cuadrática en ż
T0 = m_*om_**2*(R_**2 - z_s**2)                                     # parte sin velocidades generalizadas (arrastre)
print('T =', T_sym, '   ->   T = T2 + T0  con  T2 = m R² ż²/(R²−z²) ,  T0 = m ω² (R² − z²):', sp.simplify(T_sym - T2 - T0) == 0)

# Energía potencial: muelle (longitud = distancia entre cuentas = 2 sqrt(R²−z²)) + gravedad (en la dirección x: x1 + x2 = 0)
U_muelle = sp.Rational(1,2)*k_*(2*sp.sqrt(R_**2 - z_s**2) - 2*r0_)**2
U_grav   = sp.simplify(m_*g_*((pos1[0] + pos2[0]).subs(zf, z_s)))
print('U_gravedad = m g (x1 + x2) =', U_grav, '  (se cancela por simetría)')
U_sym = sp.simplify(U_muelle + U_grav)
E_sym = T_sym + U_sym

lag = lagrangian_hamiltonian(T_sym, U_sym, [z_s], [zd_s], [p_s])
L_sym, p_expr, H_can = lag.L, lag.p, lag.H
print('p =', p_expr)
print('H(z, p) =', H_can)

# (b) H = p²(R²−z²)/(4 m R²) + U_ef(z)  con  U_ef = U − T0
Uef_sym = 2*k_*(sp.sqrt(R_**2 - z_s**2) - r0_)**2 - m_*om_**2*(R_**2 - z_s**2)
print('H − [p²(R²−z²)/(4mR²) + U_ef] =', sp.simplify(H_can - (p_s**2*(R_**2 - z_s**2)/(4*m_*R_**2) + Uef_sym)), '   con  U_ef = 2k(√(R²−z²) − r0)² − m ω² (R²−z²)')
print('H − E =', sp.simplify(lag.H_vel - E_sym), '  = −2 T0  ->  H ≠ E: el aro gira impuesto (rheónoma). H se conserva (L no depende de t) pero E no.')

# Ecuaciones de Hamilton y de Euler-Lagrange
qdot_eq, pdot_eq = hamilton_equations(H_can, [z_s], [p_s])
print('\nż =', qdot_eq[0])
print('ṗ =', sp.simplify(pdot_eq[0]))

# (e) Ecuación de movimiento de Euler-Lagrange frente a la del enunciado
EL = euler_lagrange(L_sym, [z_s], [zd_s], [zdd_s])[0]
om0_s = sp.sqrt(2*k_/m_)
stmt  = (zdd_s + z_s/(R_**2 - z_s**2)*zd_s**2 + (om0_s**2 - om_**2)/R_**2*z_s**3 - (om0_s**2 - om_**2)*z_s
         + om0_s**2*r0_*z_s/R_**2*sp.sqrt(R_**2 - z_s**2))
print('\nz̈ + z ż²/(R²−z²) + (ω0²−ω²) z³/R² − (ω0²−ω²) z + ω0² r0 z √(R²−z²)/R² = 0   (ω0² = 2k/m)')
print('EL / (coef. de z̈) − [ecuación del enunciado] =', sp.simplify(EL/sp.diff(EL, zdd_s) - stmt))

# Poisson
print('\n{z, p} =', poisson_bracket(z_s, p_s, [z_s], [p_s]), ' ;  {H, H} =', poisson_bracket(H_can, H_can, [z_s], [p_s]), '->  H se conserva')

par_s = (m_, k_, R_, r0_, om_)
f_num   = sp.lambdify((z_s, p_s, *par_s), qdot_eq + pdot_eq, 'numpy', cse=True)    # (ż, ṗ)
H_num   = sp.lambdify((z_s, p_s, *par_s), H_can, 'numpy')
qd_num  = sp.lambdify((z_s, p_s, *par_s), lag.qdot[zd_s], 'numpy')
E_num   = sp.lambdify((z_s, zd_s, *par_s), E_sym, 'numpy')
L_num   = sp.lambdify((z_s, zd_s, *par_s), L_sym, 'numpy')
Uef_num = sp.lambdify((z_s, *par_s), Uef_sym, 'numpy')
M_num   = sp.lambdify((z_s, *par_s), 2*m_*R_**2/(R_**2 - z_s**2), 'numpy')          # p = M(z) ż

## (c), (d) EQUILIBRIOS Y ESTABILIDAD

In [ ]:
#(c), (d) Equilibrios, estabilidad y regímenes ω < ω_c  /  ω > ω_c
dUef = sp.diff(Uef_sym, z_s)
d2U  = sp.diff(Uef_sym, z_s, 2)
print("U_ef'(z) =", sp.factor(dUef), '   ->  z = 0   o   m ω² = 2k (1 − r0/√(R²−z²))')

# ρ(ω):  m ω² = 2k (1 − r0/ρ)
rho_w = sp.symbols('rho_w', positive=True)
rho_sol = sp.solve(sp.Eq(m_*om_**2, 2*k_*(1 - r0_/rho_w)), rho_w)[0]
print('\nρ(ω) =', rho_sol, '  =  r0 ω0² / (ω0² − ω²)')
print('z_eq = ± sqrt(R² − ρ(ω)²)   (existe si ρ(ω) < R)')
wc2 = sp.solve(sp.Eq(rho_sol, R_), om_**2 if False else sp.Symbol('W', positive=True))
W = sp.Symbol('W', positive=True)
wc2 = sp.solve(sp.Eq(rho_sol.subs(om_, sp.sqrt(W)), R_), W)[0]
print('ω_c² =', sp.simplify(wc2), '  = ω0² (1 − r0/R)       ->   ρ(ω) < R  <=>  ω < ω_c ;   para ω > ω_c solo queda  z = 0')

# (d) U''_ef
claim = 4*k_*r0_*(R_**2/(R_**2 - z_s**2)**sp.Rational(3,2) - 1/rho_w)
print("\nU''_ef − 4 k r0 [R²/(R²−z²)^(3/2) − 1/ρ(ω)] =", sp.simplify(d2U.subs(om_, sp.sqrt(2*k_/m_*(1 - r0_/rho_w))) - claim))
print("   en z = 0 :        U'' = 4 k r0 (1/R − 1/ρ(ω))      >0 (estable) si ρ(ω) > R (ω > ω_c) ;  <0 (inestable) si ω < ω_c")
print("   en z = ±z_eq :    U'' = 4 k r0 (R² − ρ²)/ρ³         >0  ->  estable (existe solo si ω < ω_c)")

d2U_num = sp.lambdify((z_s, *par_s), d2U, 'numpy')

def info_equilibrio(om_val):
    par = (m, k, R, r0, om_val)
    print(f'\nω = {om_val} rad/s  ({"ω < ω_c" if om_val < om_c else "ω > ω_c"},  ω_c = {om_c:.4f},  ω0 = {om0:.4f})')
    rho_ = r0*om0**2/(om0**2 - om_val**2)
    print(f"   z = 0:        U'' = {d2U_num(0.0, *par):9.4f}  ->  {'estable' if d2U_num(0.0, *par) > 0 else 'INESTABLE'}")
    if rho_ < R:
        zq = np.sqrt(R**2 - rho_**2)
        print(f"   z = ±{zq:.5f}:  U'' = {d2U_num(zq, *par):9.4f}  ->  estable   (ρ(ω) = {rho_:.4f} < R)")
    else:
        print(f'   ρ(ω) = {rho_:.4f} ≥ R  ->  no hay equilibrios ±z_eq: solo z = 0')
info_equilibrio(om_a)
info_equilibrio(om_b)

In [ ]:
#Esbozo de U_ef(z) y diagrama de bifurcación z_eq(ω)
zz = np.linspace(-R, R, 800)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))

for om_v, col in ((0.6*om_c, 'tab:green'), (om_a, 'tab:blue'), (om_c, 'k'), (om_b, 'tab:red')):
    nm = f'ω = {om_v:.2f}' + ('  (ω_c)' if om_v == om_c else '  (ω < ω_c)' if om_v < om_c else '  (ω > ω_c)')
    ax[0].plot(zz, Uef_num(zz, m, k, R, r0, om_v), color=col, lw=2, label=nm)
    rho_ = r0*om0**2/(om0**2 - om_v**2)
    if om_v < om_c:
        zq = np.sqrt(R**2 - rho_**2)
        ax[0].scatter([-zq, zq], [Uef_num(zq, m, k, R, r0, om_v)]*2, color=col, s=60, zorder=5)
ax[0].set(xlabel='z (m)', ylabel='U_ef(z) (J)', title='Potencial efectivo  U_ef = 2k(√(R²−z²) − r0)² − m ω² (R²−z²)'); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)

oms = np.linspace(0, 1.4*om_c, 600)
rho_w_ = r0*om0**2/(om0**2 - oms**2)
zq_ = np.where(rho_w_ < R, np.sqrt(np.clip(R**2 - rho_w_**2, 0, None)), np.nan)
ax[1].plot(oms, zq_, 'tab:green', lw=2.5, label='z_eq = ±√(R²−ρ²)  (estable)')
ax[1].plot(oms, -zq_, 'tab:green', lw=2.5)
ax[1].plot(oms[oms < om_c], np.zeros((oms < om_c).sum()), 'tab:red', ls='--', lw=2.5, label='z = 0  (inestable)')
ax[1].plot(oms[oms >= om_c], np.zeros((oms >= om_c).sum()), 'tab:green', lw=2.5, label='z = 0  (estable)')
ax[1].axvline(om_c, color='k', ls=':', lw=1); ax[1].text(om_c, 0.5*R, '  ω_c', fontsize=11)
ax[1].set(xlabel='ω (rad/s)', ylabel='z_eq (m)', title='Equilibrios en función de ω (bifurcación)'); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## FUNCION PARA EL MOVIMIENTO (RUNGE-KUTTA 4)

In [ ]:
def F(y, pars):
    """Segundo miembro de las ecuaciones de Hamilton. y: (2, ncasos) = (z, p). Los parámetros pueden ser arrays (uno por caso)."""
    d = f_num(*y, *pars)
    return np.array([np.broadcast_to(v, y.shape[1:]) for v in d])


def Motion(y0, pars, t):
    """Integra las ecuaciones de Hamilton con Runge-Kutta de orden 4 (varios casos a la vez)."""
    nv, nc = y0.shape
    N = len(t)
    y = np.zeros((nv, nc, N))
    y[:, :, 0] = y0

    for i in range(N - 1):
        dt = t[i+1] - t[i]
        yi = y[:, :, i]

        k1 = F(yi, pars)
        k2 = F(yi + 0.5*dt*k1, pars)
        k3 = F(yi + 0.5*dt*k2, pars)
        k4 = F(yi + dt*k3, pars)

        y[:, :, i+1] = yi + (dt/6)*(k1 + 2*k2 + 2*k3 + k4)

    return y


#Estado inicial:  p = M(z) ż  con  M(z) = 2 m R²/(R² − z²)
om_arr = np.array([cs['om'] for cs in casos])                       # ω de cada caso
z_0    = np.array([cs['z'] for cs in casos])
zd_0   = np.array([cs['zd'] for cs in casos])
pars   = (m, k, R, r0, om_arr)
y0 = np.array([z_0, M_num(z_0, *pars)*zd_0])

#Simulación
y = Motion(y0, pars, t)
z, p = y
zd = qd_num(z, p, m, k, R, r0, om_arr[:, None])
parsc = (m, k, R, r0, om_arr[:, None])                              # parámetros en columna para operar con arrays (ncasos, N)

## ENERGÍA, HAMILTONIANO Y DIAGRAMA DE FASES

In [ ]:
#Energía, Hamiltoniano y diagrama de fases (z, p)
E = E_num(z, zd, *parsc)                                   # E = T + U   (no se conserva)
H = H_num(z, p, *parsc)                                    # H(z, p)     (se conserva)
L = L_num(z, zd, *parsc)
Hr = p**2*(R**2 - z**2)/(4*m*R**2) + Uef_num(z, *parsc)    # H = p²(R²−z²)/(4mR²) + U_ef

print(f'max |H − [p²(R²−z²)/(4mR²) + U_ef]| / |H0| = {(np.abs(H - Hr).max(axis=1)/np.abs(H[:, 0])).max():.2e}')
print(f'max |H − H0| / |H0|  (todos los casos)     = {(np.abs(H - H[:, :1]).max(axis=1)/np.abs(H[:, 0])).max():.2e}   ->  H se conserva')
ig = i_gen
print(f'caso general:  E varía {np.ptp(E[ig]):.3f} J (de {E[ig].min():.3f} a {E[ig].max():.3f})  ->  E NO se conserva ;  H = {H[ig, 0]:.4f} J constante')
print(f'caso general:  z ∈ [{z[ig].min():.3f}, {z[ig].max():.3f}] m  (R = {R})   ->  pasa de un pozo al otro: {z[ig].min() < 0 < z[ig].max()}')

# --- Diagrama de fases (z, p): curvas de nivel de H para cada ω ---
cols = {i_gen: 'tab:red', i_peq: 'tab:blue', i_sup: 'tab:purple'}
nom  = {i_gen: 'movimiento general (ω < ω_c)', i_peq: 'pequeñas oscilaciones (ω < ω_c)', i_sup: 'ω > ω_c'}
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
for A, om_v, idxs, ttl in ((ax[0], om_a, (i_gen, i_peq), f'ω = {om_a} < ω_c'), (ax[1], om_b, (i_sup,), f'ω = {om_b} > ω_c')):
    zg = np.linspace(-0.995*R, 0.995*R, 500)
    pm = 1.2*max(np.abs(p[i_]).max() for i_ in idxs)
    Zg, Pg = np.meshgrid(zg, np.linspace(-pm, pm, 400))
    A.contour(Zg, Pg, H_num(Zg, Pg, m, k, R, r0, om_v), levels=35, colors='gainsboro', linewidths=0.8)
    for i_ in idxs:
        A.contour(Zg, Pg, H_num(Zg, Pg, m, k, R, r0, om_v), levels=[H[i_, 0]], colors=cols[i_], linestyles='--', linewidths=1)
        A.plot(z[i_], p[i_], color=cols[i_], lw=2, label=nom[i_])
        A.scatter(z[i_, 0], p[i_, 0], s=60, color=cols[i_], edgecolors='black', zorder=5)
    rho_ = r0*om0**2/(om0**2 - om_v**2)
    if rho_ < R:
        zq = np.sqrt(R**2 - rho_**2)
        A.scatter([-zq, zq], [0, 0], s=80, color='tab:green', zorder=5, label='equilibrio estable')
        A.scatter([0], [0], s=80, color='tab:red', marker='x', zorder=5, label='equilibrio inestable')
    else:
        A.scatter([0], [0], s=80, color='tab:green', zorder=5, label='equilibrio estable')
    A.set(xlabel='z (m)', ylabel='p (kg m²/s)', title='Diagrama de fases, ' + ttl); A.legend(fontsize=8, loc='upper right'); A.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# --- Conservación en el tiempo (caso general) ---
s = max(1, N // 20000)
fig, ax = plt.subplots(3, 1, figsize=(9, 9), sharex=True)
ax[0].plot(t[::s], E[ig, ::s], color='tab:blue', lw=2.5, label='E = T + U  (no se conserva)')
ax[0].plot(t[::s], H[ig, ::s], 'r--', lw=2, label='H = E − 2 T0  (se conserva)')
ax[0].plot(t[::s], L[ig, ::s], color='tab:purple', lw=1.2, label='L = T − U')
ax[0].set(ylabel='J', title='Caso general: E, H y L'); ax[0].legend(fontsize=8)
ax[1].plot(t[::s], z[ig, ::s], color='tab:gray', lw=1.5, label='z (m)')
ax[1].axhline(zeq_a, color='tab:green', ls=':', lw=1); ax[1].axhline(-zeq_a, color='tab:green', ls=':', lw=1, label='±z_eq')
ax[1].set(ylabel='z', title='Coordenada z(t)'); ax[1].legend(fontsize=8)
eps = 1e-18
ax[2].semilogy(t[::s], np.abs(H[ig, ::s] - H[ig, 0])/abs(H[ig, 0]) + eps, color='tab:red', lw=1, label='|H − H₀| / |H₀|  (integrador RK4)')
ax[2].set(xlabel='t (s)', ylabel='error relativo', title='Error numérico'); ax[2].legend(fontsize=8, loc='lower right')
for A in ax: A.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## PEQUEÑAS OSCILACIONES

In [ ]:
#Pequeñas oscilaciones
#   ω < ω_c, alrededor de z_eq :   Ω² = U''(z_eq)/M(z_eq) = ω0² r0 (R² − ρ²)/(R² ρ)        con  M(z_eq) = 2 m R²/ρ²
#   ω > ω_c, alrededor de z = 0:   Ω² = U''(0)/M(0) = ω² − ω_c²
Om_eq = np.sqrt(om0**2*r0*(R**2 - rho_a**2)/(R**2*rho_a))
print(f'ω = {om_a} < ω_c :  z_eq = {zeq_a:.5f} m ,  Ω = {Om_eq:.5f} rad/s  (T = {2*np.pi/Om_eq:.4f} s)')
print(f'ω = {om_b} > ω_c :  z = 0 ,  Ω = sqrt(ω² − ω_c²) = {np.sqrt(om_b**2 - om_c**2):.5f} rad/s  (T = {2*np.pi/np.sqrt(om_b**2 - om_c**2):.4f} s)')

delta0s  = z_0[:4] - zeq_a
delta    = z[:4] - zeq_a
delta_an = delta0s[:, None]*np.cos(Om_eq*t)[None, :]
print(f'\n{"δ0":>8}{"max|δ−δ_an|/δ0":>18}')
for i_ in range(4):
    print(f'{delta0s[i_]:>8.4f}{np.abs(delta[i_] - delta_an[i_]).max()/delta0s[i_]:>18.2e}')

s = max(1, N // 5000)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(t[::s], z[i_peq, ::s], lw=2, label='numérica')
ax[0].plot(t[::s], (zeq_a + delta_an[i_peq])[::s], 'r--', lw=1.2, label='z_eq + δ0 cos(Ω t)')
ax[0].axhline(zeq_a, color='k', ls=':', lw=1, label='z_eq')
ax[0].set(xlabel='t (s)', ylabel='z (m)', title=f'Pequeñas oscilaciones (ω = {om_a}, δ0 = {delta0s[i_peq]})'); ax[0].legend(fontsize=8)
ax[1].plot(t[::s], (delta[i_peq] - delta_an[i_peq])[::s], color='tab:green')
ax[1].set(xlabel='t (s)', ylabel='δ_num − δ_an', title='Diferencia (desfase por no linealidad)')
for A in ax: A.grid(alpha=0.3)
plt.tight_layout(); plt.show()


#Frecuencia de las pequeñas oscilaciones para distintos ω (numérica frente a fórmula)
oms_s = np.array([0.5, 1.5, 2.5, 3.0, 3.5, 4.5, 6.0])
rho_s = r0*om0**2/(om0**2 - oms_s**2)
z_eq_s = np.where(rho_s < R, np.sqrt(np.clip(R**2 - rho_s**2, 0, None)), 0.0)          # z_eq (ω < ω_c)  o  0 (ω > ω_c)
Om_teo = np.where(oms_s < om_c, np.sqrt(om0**2*r0*(R**2 - np.minimum(rho_s, R)**2)/(R**2*np.minimum(rho_s, R))),
                  np.sqrt(np.clip(oms_s**2 - om_c**2, 0, None)))
t_s  = np.arange(0, 12, 1e-3)
y0_s = np.array([z_eq_s + 0.01, np.zeros_like(oms_s)])
z_s_ = Motion(y0_s, (m, k, R, r0, oms_s), t_s)[0]

def omega_numerica(x, tt):
    """Frecuencia a partir de los cruces ascendentes de x(t) por cero (interpolación lineal)."""
    cr = np.where((x[:-1] < 0) & (x[1:] >= 0))[0]
    tc = tt[cr] - x[cr]*(tt[cr+1] - tt[cr])/(x[cr+1] - x[cr])
    return 2*np.pi/np.mean(np.diff(tc))

print(f'\n{"ω":>6}{"régimen":>10}{"punto de equilibrio":>22}{"Ω teórica":>12}{"Ω numérica":>13}')
for j, om_v in enumerate(oms_s):
    reg = 'ω < ω_c' if om_v < om_c else 'ω > ω_c'
    pto = f'z = ±{z_eq_s[j]:.4f}' if om_v < om_c else 'z = 0'
    print(f'{om_v:>6.2f}{reg:>10}{pto:>22}{Om_teo[j]:>12.4f}{omega_numerica(z_s_[j] - z_eq_s[j], t_s):>13.4f}')

## ANIMACIÓN

In [ ]:
#Animación: función que anima un caso (vista 3D, aro, vista cenital y potencial efectivo) y se aplica a las tres situaciones
from IPython.display import display
plt.rcParams['animation.embed_limit'] = 100        # MB; el límite por defecto (20) se queda corto

interactivo = True     # True : figuras interactivas, se gira la vista 3D con el ratón mientras se anima (requiere ipympl)
                       # False: animación HTML (el ratón no gira la vista; usa 'rotar' para que la cámara gire sola)
elev, azim  = 20, -60  # ángulos iniciales de la vista 3D (grados)
rotar       = 0        # giro automático de la cámara (grados por segundo; 0 = fija). Funciona en los dos modos

T_anim  = 6        # segundos simulados a animar (máx. tf)
fps     = 12       # fotogramas por segundo
speed   = 1.0      # 1 = tiempo real, 0.5 = cámara lenta
trail_s = 1.5      # duración de la estela de las cuentas (s); 0 = sin estela
n_coils = 12       # vueltas del muelle dibujado

if interactivo:
    try:
        get_ipython().run_line_magic('matplotlib', 'widget')      # backend interactivo:  pip install ipympl
    except Exception as e:
        interactivo = False
        print('Recordatorio para el modo interactivo (instalar ipympl).', e)


def animate_hoop(k_caso, titulo='', T_anim=T_anim, fps=fps, speed=speed, trail_s=trail_s, elev=elev, azim=azim, rotar=rotar):
    """Crea la figura y la animación del caso 'k_caso' (índice en 'casos'). Devuelve (fig, anim)."""
    step    = max(1, round(speed / (fps * dt)))
    idx     = np.arange(0, min(N, int(T_anim / dt)), step)
    trail_n = int(trail_s / (dt * step))

    om_v = om_arr[k_caso]
    ts, zs = t[idx], z[k_caso, idx]
    rs  = np.sqrt(R**2 - zs**2)                                     # distancia de las cuentas al eje
    phi = om_v*ts                                                   # giro del aro
    c, s_ = np.cos(phi), np.sin(phi)
    x1, y1, x2, y2 = rs*c, rs*s_, -rs*c, -rs*s_                     # cuentas 1 y 2 (lados opuestos)
    H0 = H[k_caso, 0]
    zg_ = np.linspace(-R, R, 600)
    Uef_ = Uef_num(zg_, m, k, R, r0, om_v)
    al = np.linspace(0, 2*np.pi, 200)
    amp = 0.035*R

    def spring_pts(xa, ya, xb, yb, za, n=n_coils):
        """Zigzag entre las dos cuentas (desplazamiento en la dirección z)."""
        sg = np.linspace(0, 1, 2*n + 1)
        off = amp*np.where(np.arange(2*n + 1) % 2 == 1, 1.0, -1.0); off[0] = off[-1] = 0
        return xa + sg*(xb - xa), ya + sg*(yb - ya), za + off

    fig = plt.figure(figsize=(11, 8.4))
    ax3 = fig.add_subplot(2, 2, 1, projection='3d')
    axa = fig.add_subplot(2, 2, 2)                     # plano del aro (ρ-z), girando con él
    axt = fig.add_subplot(2, 2, 3)                     # plano cenital (x-y)
    axu = fig.add_subplot(2, 2, 4)                     # potencial efectivo
    fig.subplots_adjust(left=0.07, right=0.97, bottom=0.07, top=0.90, wspace=0.28, hspace=0.3)
    fig.suptitle(titulo, fontsize=12, y=0.985)

    # --- Vista 3D ---
    L3 = 1.15*R
    ax3.set(xlim=(-L3, L3), ylim=(-L3, L3), zlim=(-L3, L3), xlabel='x', ylabel='y', zlabel='z', title='Vista 3D (eje de giro: z)')
    ax3.set_box_aspect((1, 1, 1)); ax3.grid(False); ax3.view_init(elev=elev, azim=azim)
    ax3.plot([0, 0], [0, 0], [-L3, L3], 'k:', lw=1)
    hoop3, = ax3.plot([], [], [], '-', color='dimgray', lw=2)
    spr3,  = ax3.plot([], [], [], '-', color='tab:orange', lw=1.3)
    b3,    = ax3.plot([], [], [], 'o', color='tab:red', mec='black', ms=9)
    tr3,   = ax3.plot([], [], [], '-', color='tab:red', lw=1, alpha=0.6)

    # --- Plano del aro (ρ-z) ---
    axa.set(xlim=(-1.15*R, 1.15*R), ylim=(-1.15*R, 1.15*R), aspect='equal', xlabel='ρ (m)', ylabel='z (m)', title='Aro (plano que gira con él)')
    axa.plot(R*np.cos(al), R*np.sin(al), 'k-', lw=1.5); axa.axvline(0, color='gray', lw=0.5); axa.axhline(0, color='gray', lw=0.5)
    axa.grid(alpha=0.3)
    bA,   = axa.plot([], [], 'o', color='tab:red', mec='black', ms=10)
    sprA, = axa.plot([], [], '-', color='tab:orange', lw=1.5)
    trA,  = axa.plot([], [], '-', color='tab:red', lw=1, alpha=0.6)

    # --- Plano cenital (x-y) ---
    axt.set(xlim=(-1.15*R, 1.15*R), ylim=(-1.15*R, 1.15*R), aspect='equal', xlabel='x (m)', ylabel='y (m)', title='Plano cenital (x-y)')
    axt.plot(R*np.cos(al), R*np.sin(al), color='lightgray', lw=1); axt.plot(0, 0, 'ko', ms=4); axt.grid(alpha=0.3)
    hoopT, = axt.plot([], [], '-', color='dimgray', lw=2)
    bT,    = axt.plot([], [], 'o', color='tab:red', mec='black', ms=9)
    trT,   = axt.plot([], [], '-', color='tab:red', lw=1, alpha=0.6)

    # --- Potencial efectivo ---
    axu.plot(zg_, Uef_, 'k', lw=2)
    axu.axhline(H0, color='tab:blue', ls='--', lw=1, label='H (energía total del problema reducido)')
    axu.set(xlabel='z (m)', ylabel='U_ef (J)', title=f'Potencial efectivo (ω = {om_v:g} rad/s)'); axu.grid(alpha=0.3); axu.legend(fontsize=7, loc='upper center')
    uD, = axu.plot([], [], 'o', color='tab:red', mec='black', ms=10, zorder=5)

    time_txt = fig.text(0.02, 0.945, '')

    def update(k_):
        j = max(0, k_ - trail_n)
        a_ = al[:]                                                                    # aro en 3D: plano que contiene a z y a e_ρ(φ)
        hoop3.set_data_3d(R*np.cos(a_)*c[k_], R*np.cos(a_)*s_[k_], R*np.sin(a_))
        sx, sy, sz = spring_pts(x1[k_], y1[k_], x2[k_], y2[k_], zs[k_])
        spr3.set_data_3d(sx, sy, sz)
        b3.set_data_3d([x1[k_], x2[k_]], [y1[k_], y2[k_]], [zs[k_], zs[k_]])
        tr3.set_data_3d(x1[j:k_+1], y1[j:k_+1], zs[j:k_+1])

        bA.set_data([rs[k_], -rs[k_]], [zs[k_], zs[k_]])
        sxa, sza = np.linspace(-rs[k_], rs[k_], 2*n_coils + 1), zs[k_] + amp*np.where(np.arange(2*n_coils + 1) % 2 == 1, 1.0, -1.0)
        sza[0] = sza[-1] = zs[k_]
        sprA.set_data(sxa, sza)
        trA.set_data(rs[j:k_+1], zs[j:k_+1])

        hoopT.set_data([-R*c[k_], R*c[k_]], [-R*s_[k_], R*s_[k_]])
        bT.set_data([x1[k_], x2[k_]], [y1[k_], y2[k_]])
        trT.set_data(x1[j:k_+1], y1[j:k_+1])

        uD.set_data([zs[k_]], [Uef_num(zs[k_], m, k, R, r0, om_v)])

        if rotar != 0:                                 # solo si se pide giro automático (si no, se respeta la vista del ratón)
            ax3.view_init(elev=elev, azim=azim + rotar*ts[k_])
        time_txt.set_text(f't = {ts[k_]:.2f} s    z = {zs[k_]:.3f} m    r = {rs[k_]:.3f} m    φ = {np.degrees(phi[k_]) % 360:.0f}°')
        return hoop3, spr3, b3, tr3, bA, sprA, trA, hoopT, bT, trT, uD, time_txt

    anim = animation.FuncAnimation(fig, update, frames=len(idx), interval=1000/fps, blit=False)   # blit=False: necesario con ejes 3D
    return fig, anim


# Se aplica a las situaciones iniciales
situaciones = [(i_gen, f'Situación 1: movimiento general (ω = {om_a} < ω_c = {om_c:.2f})'),
               (i_peq, f'Situación 2: pequeñas oscilaciones alrededor de z_eq (ω = {om_a} < ω_c)'),
               (i_sup, f'Situación 3: ω = {om_b} > ω_c  (oscilación alrededor de z = 0)')]
anims = [animate_hoop(k_caso, titulo) for k_caso, titulo in situaciones]      # se guardan las referencias para que no se detengan

if interactivo:
    plt.show()                                     # no cerrar las figuras: son las que se manejan con el ratón
else:
    for fig_, anim_ in anims:
        plt.close(fig_)
        display(HTML(anim_.to_jshtml()))